# EP03 — LLM-Based Platform for Science (Hydropower)
**COMPSCI 713 · S1 2024 Q3 · 10 marks**

Write a concise technical route for building an LLM platform for hydropower station location planning — starting from data collection.

---

## Full Answer — Technical Route [10 marks]

### Step 1 — Domain Data Collection
Gather multi-source, domain-specific data:
- **Text:** hydrology research papers, engineering reports, environmental impact assessments,   regulatory documents, geographic surveys
- **Structured:** GIS datasets (river flow rates, topography, precipitation, population density),   grid infrastructure maps
- **Historical:** records of existing hydropower sites, performance data, failure/risk cases

### Step 2 — Data Preprocessing & Tokenisation
- Clean, deduplicate, normalise text; remove low-quality web noise
- Domain-aware tokenisation (handle units: MW, m3/s, km2, GPS coordinates)
- Convert structured/tabular data to natural language descriptions:
  `"River X: mean annual flow = 450 m3/s, catchment area = 2200 km2, gradient = 3.2%"`

### Step 3 — Pre-training or Foundation Model Fine-Tuning
**Option A (from scratch):** Train a Transformer (GPT-style decoder or T5-style encoder-decoder) on the domain corpus using causal language modelling or masked LM objectives.

**Option B (practical):** Start from a public foundation model (LLaMA, GPT family) and continue pre-training on the hydrology corpus — injects domain vocabulary and facts at far lower compute cost.

### Step 4 — Supervised Fine-Tuning on Downstream Tasks
Create instruction datasets from expert knowledge:
- **Site suitability scoring:** given river/terrain data -> rating + reasoning
- **Q&A pairs:** expert questions about siting criteria with validated answers
- **Report generation:** structured data input -> professional recommendation text

Fine-tune with SFT (supervised fine-tuning); optionally apply RLHF if expert feedback is available.

### Step 5 — Retrieval-Augmented Generation (RAG)
Pair the LLM with a vector database of up-to-date documents (new environmental regulations, recent climate studies). At inference, retrieve relevant chunks and inject into context — reduces hallucination and keeps knowledge current without retraining.

### Step 6 — Evaluation
- Benchmark on held-out site-recommendation tasks with expert-labelled ground truth
- Check factual accuracy against known hydropower databases
- Human expert review for regulatory/environmental bias

### Step 7 — Deployment
Serve as an API connected to a GIS front-end; engineers query by region/river and receive structured siting recommendations with cited evidence and confidence scores.

---

## Code Demo — Visualising a RAG Pipeline (conceptual)

In [ ]:
# ─── Conceptual RAG pipeline — retrieval + generation ────────────────────────
# In production: replace MockEmbedder/MockLLM with real models.

import numpy as np

# ── Mock document store (would be a vector DB in production) ──────────────────
DOCS = [
    {'id': 1, 'text': 'River Waikato: mean flow 330 m3/s, gradient 1.8%, suitable for run-of-river.'},
    {'id': 2, 'text': 'Environmental regulations require 30% minimum flow maintenance at all times.'},
    {'id': 3, 'text': 'Manapouri power station uses lake storage; capacity 850 MW, head 178m.'},
    {'id': 4, 'text': 'Seismic risk assessment is mandatory for sites in Canterbury and Otago.'},
    {'id': 5, 'text': 'Run-of-river plants are preferred in ecologically sensitive areas.'},
]

def mock_embed(text):
    'Fake embedder: hash words into a 8-dim vector.'
    np.random.seed(abs(hash(text)) % 2**31)
    return np.random.randn(8)

doc_vecs = np.array([mock_embed(d['text']) for d in DOCS])

def retrieve(query, top_k=2):
    q_vec = mock_embed(query)
    # Cosine similarity
    sims = doc_vecs @ q_vec / (np.linalg.norm(doc_vecs, axis=1) * np.linalg.norm(q_vec) + 1e-9)
    top = np.argsort(sims)[::-1][:top_k]
    return [DOCS[i] for i in top], sims[top]

def rag_answer(query):
    docs, scores = retrieve(query)
    context = '\n'.join(f"[Doc {d['id']}] {d['text']}" for d in docs)
    # In production: pass context + query to LLM
    print(f'Query: {query}')
    print(f'Retrieved context:')
    for d, s in zip(docs, scores):
        print(f'  (sim={s:.3f}) {d["text"]}')
    print(f'[LLM would generate answer grounded in above context]\n')

rag_answer('What environmental rules apply to new hydropower sites?')
rag_answer('Is the Waikato River suitable for a run-of-river plant?')